# XAct.jl — Getting Started

> **Note:** Auto-generated from `notebooks/julia/basics.qmd` via Quarto.
> Edit the source `.qmd` file, not the `.ipynb`.

## Overview

This notebook walks through the core workflow of `XAct.jl`:
defining a spacetime manifold, adding a metric, computing curvature tensors,
and simplifying expressions using the Butler-Portugal canonicalization algorithm.

Expressions are written using the **typed API** — index objects and operator
overloading — which validates slot counts and manifold membership at
construction time rather than deep inside the engine.

**Google Colab:** select *Runtime → Change runtime type → Julia*.

## 1. Setup

If running on Google Colab or a fresh environment, install the package first.
In the Docker image or a local dev checkout this cell is a no-op.

In [ ]:
# Uncomment the lines below if running on Google Colab:
# using Pkg
# Pkg.add("XAct")

## 2. Setup

In [ ]:
using XAct

## 2. Define a Manifold

A 4-dimensional spacetime manifold $M$ with abstract indices $a, b, c, d, e, f$.

`@indices` declares typed index variables bound to the manifold — `-a` then
gives a covariant (down) index.

In [ ]:
reset_state!()
M = def_manifold!(:M, 4, [:a, :b, :c, :d, :e, :f])
@indices M a b c d e f

## 3. Define a Metric

The metric $g_{ab}$ with Lorentzian signature $(-,+,+,+)$ and covariant
derivative $\nabla$ (called `CD` internally).
This automatically creates Riemann, Ricci, RicciScalar, Weyl, Einstein, and
Christoffel tensors.

In [ ]:
g = def_metric!(-1, "g[-a,-b]", :CD)

# Grab tensor handles for use throughout the notebook.
Riem = tensor(:RiemannCD)
Ric  = tensor(:RicciCD)
g_h  = tensor(:g)

## 4. Canonicalization

The Butler-Portugal algorithm brings tensor expressions into a canonical form.
Build expressions with `[]` — the tensor handle validates the slot count
and manifold membership immediately.

Symmetric metric: $g_{ba} - g_{ab} = 0$:

In [ ]:
ToCanonical(g_h[-b,-a] - g_h[-a,-b])

Define a symmetric rank-2 tensor $T_{ab}$ (e.g. the energy-momentum tensor)
and verify symmetry:

In [ ]:
def_tensor!(:T, ["-a", "-b"], :M; symmetry_str="Symmetric[{-a,-b}]")
T_h = tensor(:T)
ToCanonical(T_h[-b,-a] - T_h[-a,-b])

> **String API:** `ToCanonical("T[-b,-a] - T[-a,-b]")` is equivalent.
> Strings still work everywhere; the typed API adds upfront validation.

## 5. Contraction

`Contract` lowers/raises indices via the metric. Define a vector $V^a$
and contract with the metric to get $V_b$:

In [ ]:
def_tensor!(:V, ["a"], :M)
V_h = tensor(:V)
Contract(V_h[a] * g_h[-a,-b])

## 6. Riemann Tensor Identities

The Riemann tensor has well-known symmetries that the canonicalizer
automatically recognizes.

First Bianchi identity — $R_{abcd} + R_{acdb} + R_{adbc} = 0$:

In [ ]:
ToCanonical(Riem[-a,-b,-c,-d] + Riem[-a,-c,-d,-b] + Riem[-a,-d,-b,-c])

Antisymmetry in the first pair — $R_{abcd} + R_{bacd} = 0$:

In [ ]:
ToCanonical(Riem[-a,-b,-c,-d] + Riem[-b,-a,-c,-d])

Pair symmetry — $R_{abcd} = R_{cdab}$:

In [ ]:
ToCanonical(Riem[-a,-b,-c,-d] - Riem[-c,-d,-a,-b])

## 7. Perturbation Theory

Define a perturbation tensor $h_{ab}$ (symmetric, same slots as the metric)
and perturb the metric to first order:

In [ ]:
def_tensor!(:h, ["-a", "-b"], :M; symmetry_str="Symmetric[{-a,-b}]")
def_perturbation!(:h, :g, 1)
perturb(g_h[-a,-b], 1)

## 8. Validation

The typed API catches mistakes at construction time:

In [ ]:
# Wrong slot count — raised before reaching the engine
try
    Riem[-a,-b]     # ERROR: RiemannCD has 4 slots, got 2
catch e
    println(e)
end

## Next Steps

- **Coordinate components (xCoba):** assign a basis, compute Christoffel symbols numerically
- **Covariant derivatives:** commute CovDs, integration by parts
- **Invariant simplification:** `RiemannSimplify` for scalar polynomial identities
- Full documentation: [saxa.xyz/XAct.jl](../index.md)